# Goal #7 — Topic Modeling by Primary Major Class (Scopus + Patents, Abstract-only)

Goal: use topic structure to *organically* separate Scopus vs patents **inside** each major class (i.e., different topic prevalence by source), and export per-document topic assignments as a **sub-classification** within each major class.

Inputs (from Goal #6):
- `outputs/goal6/goal6_scopus_major_classified.csv`
- `outputs/goal6/goal6_patents_major_classified.csv`

CSV outputs (written under `outputs/goal7/`):
- `goal7_scopus_topic_subclassified.csv`
- `goal7_patents_topic_subclassified.csv`


In [ ]:
# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/mani

In [ ]:
# Runtime setup (Colab-only: installs, paths, reproducibility)

import re
import json
import sys
import math
import hashlib
import unicodedata
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd

try:
    import google.colab  # type: ignore
except Exception as e:
    raise RuntimeError(
        'This notebook is Colab-only. Open it in Google Colab and run the Drive mount cell first.'
    ) from e


def pip_install(*args: str) -> None:
    cmd = [sys.executable, '-m', 'pip', 'install', '-q', *args]
    subprocess.check_call(cmd)


# Install dependencies (Colab)
SENTINEL = Path('/content/.mani_goal7_deps_ok')
if not SENTINEL.exists():
    pip_install('bertopic', 'sentence-transformers', 'umap-learn', 'hdbscan')
    pip_install('seaborn', 'plotly', 'kaleido')
    SENTINEL.write_text('ok', encoding='utf-8')

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import NMF, TruncatedSVD, LatentDirichletAllocation

from sentence_transformers import SentenceTransformer
from bertopic import BERTopic

# Reproducibility
SEED = 421
np.random.seed(SEED)

# Project root (Drive)
ROOT = Path('/content/drive/MyDrive/mani').resolve()
if not ROOT.exists():
    raise FileNotFoundError(f"Drive project root not found: {ROOT} (did Drive mount succeed?)")

OUT_DIR = ROOT / 'outputs' / 'goal7'
OUT_DIR.mkdir(parents=True, exist_ok=True)

CONFIG = {
    'scopus_goal6_csv': str(ROOT / 'outputs' / 'goal6' / 'goal6_scopus_major_classified.csv'),
    'patents_goal6_csv': str(ROOT / 'outputs' / 'goal6' / 'goal6_patents_major_classified.csv'),
    'out_dir': str(OUT_DIR),

    # Corpus construction (ABSTRACT-ONLY)
    'min_chars': 80,
    'min_tokens': 15,

    # Stratification filters
    'min_docs_total': 100,
    'min_docs_per_source': 25,

    # TF-IDF baseline (Section 5)
    'tfidf_max_features': 20000,
    'tfidf_min_df': 2,
    'tfidf_max_df': 0.95,
    'baseline_max_points': 4000,

    # NMF (used for optional quick baseline / debugging)
    'nmf_n_topics': 8,
    'nmf_top_words': 12,

    # BERTopic
    'embedding_model': 'all-MiniLM-L6-v2',
    'bertopic_top_n_words': 12,
    'bertopic_min_topic_size': 25,
    'bertopic_n_gram_range': (1, 2),

    # Separation metrics
    'jsd_epsilon': 1e-9,
    'dominance_threshold': 0.65,

    # Optional LDA
    'run_lda_baseline': False,
    'lda_n_topics': 8,
}

print('ROOT:', ROOT)
print('OUT_DIR:', OUT_DIR)
print(json.dumps(CONFIG, indent=2))


## 1) Load Patent + Scopus CSVs (Goal #7 inputs)

Read the two Goal #6 classified CSVs from disk (patents + scopus), validate schemas, and preview content.

In [ ]:
from typing import Iterable


def _require_cols(df: pd.DataFrame, cols: list[str], label: str) -> None:
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise ValueError(f"{label}: missing required columns: {missing}")


def _read_csv_or_fail(path_str: str, label: str) -> pd.DataFrame:
    path = Path(path_str)
    if not path.exists():
        raise FileNotFoundError(f"{label}: missing file: {path}")
    df = pd.read_csv(path, dtype=str)
    if len(df) == 0:
        raise ValueError(f"{label}: file is empty: {path}")
    return df


scopus_path = CONFIG['scopus_goal6_csv']
patents_path = CONFIG['patents_goal6_csv']

scopus_raw = _read_csv_or_fail(scopus_path, label='Scopus (Goal6)')
patents_raw = _read_csv_or_fail(patents_path, label='Patents (Goal6)')

required = ['Serial Number', 'Title', 'Abstract', 'Primary Major Class', 'Primary Major Label', 'Primary Major Score']
_require_cols(scopus_raw, required + ['Year'], label='Scopus (Goal6)')
_require_cols(patents_raw, required + ['Year'], label='Patents (Goal6)')

print('Scopus shape:', scopus_raw.shape)
print('Patents shape:', patents_raw.shape)

display(scopus_raw[required + ['Year']].head(3))
display(patents_raw[required + ['Year']].head(3))

print('Scopus columns:', len(scopus_raw.columns))
print('Patents columns:', len(patents_raw.columns))
scopus_raw.info()
patents_raw.info()

## 2) Normalize Schemas & Build a Unified Corpus (patents vs scopus)

Create a unified DataFrame with consistent fields and a single text column for modeling.

- `doc_id`: stable id (source + serial)
- `source`: `patent` or `scopus`
- `year`
- `major_class`, `major_label`, `major_score`
- `text_raw`: **Abstract only** (for both patents and scopus)

Also: drop/flag missing/short text and deduplicate exact normalized text hashes.

In [ ]:
def _as_int_series(s: pd.Series) -> pd.Series:
    return pd.to_numeric(s, errors='coerce').astype('Int64')


def _normalize_ws(s: str) -> str:
    return re.sub(r'\s+', ' ', (s or '').strip())


def _hash_text(s: str) -> str:
    # Exact hash for fast dedupe ("near-identical" here means identical after normalization)
    return hashlib.sha256(s.encode('utf-8', errors='ignore')).hexdigest()


def build_unified_corpus(scopus_df: pd.DataFrame, patents_df: pd.DataFrame) -> pd.DataFrame:
    sc = scopus_df.copy()
    pa = patents_df.copy()

    sc['source'] = 'scopus'
    pa['source'] = 'patent'

    # Standard fields
    for df in (sc, pa):
        df['year'] = _as_int_series(df['Year'])
        df['major_class'] = _as_int_series(df['Primary Major Class'])
        df['major_label'] = df['Primary Major Label'].fillna('').astype(str)
        df['major_score'] = pd.to_numeric(df['Primary Major Score'], errors='coerce')
        df['title'] = df['Title'].fillna('').astype(str)
        df['abstract'] = df['Abstract'].fillna('').astype(str)
        df['serial'] = df['Serial Number'].fillna('').astype(str)
        df['doc_id'] = df['source'].astype(str) + ':' + df['serial'].astype(str)

    # Build text_raw (ABSTRACT-ONLY for topic modeling)
    # Keep title column for reference/export, but do not include it in the modeling corpus.
    pa['text_raw'] = pa['abstract'].str.strip()
    sc['text_raw'] = sc['abstract'].str.strip()

    keep_cols = [
        'doc_id', 'source', 'year',
        'major_class', 'major_label', 'major_score',
        'serial',
        'title', 'abstract',
        'text_raw',
    ]

    corpus = pd.concat([sc[keep_cols], pa[keep_cols]], ignore_index=True)

    # Basic drop/flags
    corpus['text_raw_norm'] = corpus['text_raw'].map(_normalize_ws)
    corpus['text_hash'] = corpus['text_raw_norm'].map(_hash_text)

    before = len(corpus)
    corpus = corpus[corpus['text_raw_norm'].str.len() > 0].copy()
    print('Dropped empty text rows:', before - len(corpus))

    before = len(corpus)
    corpus = corpus.drop_duplicates(subset=['text_hash']).copy()
    print('Dropped exact-duplicate rows:', before - len(corpus))

    return corpus


df_corpus = build_unified_corpus(scopus_raw, patents_raw)
print('Unified corpus rows:', len(df_corpus))
display(df_corpus.head(5))

# Quick sanity
print(df_corpus[['source', 'major_class']].value_counts().head(10))

## 3) Text Cleaning Pipeline (abstract-only; stopwords; n-grams)

Implement reusable preprocessing functions and produce:
- `text_clean`
- length fields
- configurable domain stopwords

Also plot text-length distributions by source.

In [ ]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# Optional lemmatization (disabled by default)
USE_LEMMATIZATION = bool(CONFIG.get('use_lemmatization', False))
if USE_LEMMATIZATION:
    pip_install('nltk')
    import nltk
    from nltk.stem import WordNetLemmatizer
    nltk.download('wordnet')
    nltk.download('omw-1.4')
    _lemm = WordNetLemmatizer()
else:
    _lemm = None


BOILERPLATE_ABSTRACTS = {
    '[no abstract available]': '',
    'no abstract available': '',
}

DOMAIN_STOPWORDS_COMMON = {
    # Generic technical boilerplate
    'device', 'method', 'system', 'apparatus', 'use', 'using', 'used', 'provide', 'provided',
    'includes', 'including', 'described', 'present', 'invention', 'disclosure',
}

DOMAIN_STOPWORDS_BY_SOURCE = {
    'patent': {'claim', 'claims', 'embodiment', 'preferred', 'wherein'},
    'scopus': {'study', 'results', 'paper', 'article'},
}


def clean_text(text: str, *, source: str) -> str:
    t = (text or '').strip()

    # Unicode normalize
    t = unicodedata.normalize('NFKC', t)

    # Lowercase
    t = t.lower()

    # Remove boilerplate abstracts
    t = BOILERPLATE_ABSTRACTS.get(t, t)

    # Normalize whitespace
    t = re.sub(r'\s+', ' ', t).strip()

    # Keep letters/numbers, collapse punctuation to spaces
    t = re.sub(r"[^a-z0-9\s]", " ", t)
    t = re.sub(r'\s+', ' ', t).strip()

    if USE_LEMMATIZATION and _lemm is not None:
        toks = [
            _lemm.lemmatize(tok)
            for tok in t.split()
            if tok
        ]
        t = ' '.join(toks)

    # Remove very short tokens
    t = ' '.join([tok for tok in t.split() if len(tok) >= 3])

    # Domain stopwords (applied later via vectorizers too; doing a light pass here helps hashing)
    stop = set(ENGLISH_STOP_WORDS) | set(DOMAIN_STOPWORDS_COMMON) | set(DOMAIN_STOPWORDS_BY_SOURCE.get(source, set()))
    t = ' '.join([tok for tok in t.split() if tok not in stop])

    return t.strip()


def add_clean_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out['text_clean'] = [clean_text(t, source=s) for t, s in zip(out['text_raw'].tolist(), out['source'].tolist())]
    out['n_chars'] = out['text_clean'].str.len()
    out['n_tokens'] = out['text_clean'].str.split().map(len)
    return out


df_corpus = add_clean_columns(df_corpus)

# Drop rows with missing/short text
before = len(df_corpus)
df_corpus = df_corpus[(df_corpus['n_chars'] >= int(CONFIG['min_chars'])) & (df_corpus['n_tokens'] >= int(CONFIG['min_tokens']))].copy()
print('Dropped short-text rows:', before - len(df_corpus))

# Length distributions by source
plt.figure(figsize=(10, 4))
sns.histplot(data=df_corpus, x='n_tokens', hue='source', bins=60, element='step', stat='density', common_norm=False)
plt.title('Token length distribution (text_clean) by source')
plt.xlim(0, min(400, int(df_corpus['n_tokens'].quantile(0.99)) if len(df_corpus) else 400))
plt.show()

display(df_corpus[['doc_id', 'source', 'year', 'major_class', 'major_label', 'n_tokens', 'text_clean']].head(5))

## 4) Stratify by Primary Major Class (label + code) & Basic Stats

Group by `major_class`/`major_label`, compute counts per class split by source, and filter to classes with sufficient documents for topic modeling.

In [ ]:
# Counts by class and source
counts = (
    df_corpus
    .groupby(['major_class', 'major_label', 'source'])
    .size()
    .rename('count')
    .reset_index()
)

pivot = counts.pivot_table(index=['major_class', 'major_label'], columns='source', values='count', fill_value=0)
pivot['total'] = pivot.sum(axis=1)

pivot = pivot.sort_values('total', ascending=False)
display(pivot.reset_index())

min_total = int(CONFIG['min_docs_total'])
min_per_source = int(CONFIG['min_docs_per_source'])

retained = pivot[(pivot['total'] >= min_total) & (pivot.get('patent', 0) >= min_per_source) & (pivot.get('scopus', 0) >= min_per_source)].copy()

retained_classes = retained.reset_index()[['major_class', 'major_label']].to_dict('records')
print('Retained classes:', len(retained_classes))
display(retained.reset_index())

if len(retained_classes) == 0:
    print('No classes meet thresholds. Consider lowering CONFIG[min_docs_total] / CONFIG[min_docs_per_source].')

## 5) Vectorization Baselines (TF-IDF) + Dimensionality Reduction

Build a baseline TF‑IDF representation and a simple 2D projection to quickly inspect whether Scopus vs patents separate inside each major class (pre-topic sanity check).

In [ ]:
from sklearn.utils import shuffle


def tfidf_svd_scatter(df_class: pd.DataFrame, *, title: str) -> None:
    texts = df_class['text_clean'].tolist()
    sources = df_class['source'].tolist()

    if len(texts) < 10:
        print(f"Skip baseline scatter (too few docs): {title} | n={len(texts)}")
        return

    # Subsample for faster plotting
    max_points = int(CONFIG['baseline_max_points'])
    if len(df_class) > max_points:
        df_plot = df_class.sample(n=max_points, random_state=SEED)
    else:
        df_plot = df_class

    vect = TfidfVectorizer(
        max_features=int(CONFIG['tfidf_max_features']),
        min_df=int(CONFIG['tfidf_min_df']),
        max_df=float(CONFIG['tfidf_max_df']),
        stop_words='english',
        ngram_range=(1, 2),
    )

    X = vect.fit_transform(df_plot['text_clean'].tolist())

    svd = TruncatedSVD(n_components=2, random_state=SEED)
    X2 = svd.fit_transform(X)

    plot_df = pd.DataFrame({'x': X2[:, 0], 'y': X2[:, 1], 'source': df_plot['source'].tolist()})

    plt.figure(figsize=(6, 5))
    sns.scatterplot(data=plot_df, x='x', y='y', hue='source', alpha=0.6, s=18)
    plt.title(title)
    plt.tight_layout()
    plt.show()


for row in retained_classes[:5]:
    code = int(row['major_class'])
    label = row['major_label']
    df_class = df_corpus[df_corpus['major_class'] == code].copy()
    tfidf_svd_scatter(df_class, title=f"TF-IDF+SVD (Major {code} — {label})")

print('Baseline scatter shown for up to first 5 retained classes.')

## 6) Topic Modeling per Major Class (BERTopic on combined sources)

For each retained major class, fit a BERTopic model on the combined corpus (patents + scopus together) while retaining metadata (`source`, `year`). Save per-doc topic assignments and per-topic top terms; keep a model registry keyed by class.

In [ ]:
from umap import UMAP
import hdbscan


def _topic_top_terms(topic_model: BERTopic, topic_id: int, top_n: int) -> str:
    words = topic_model.get_topic(topic_id) or []
    return ', '.join([w for w, _ in words[:top_n]])


def _safe_max_topic_prob(probs_row) -> float | None:
    if probs_row is None:
        return None
    try:
        return float(np.max(probs_row))
    except Exception:
        return None


embedder = SentenceTransformer(CONFIG['embedding_model'])

class_models: dict[int, BERTopic] = {}
class_docs: dict[int, pd.DataFrame] = {}
class_embeddings: dict[int, np.ndarray] = {}

per_doc_rows: list[dict] = []
per_topic_rows: list[dict] = []

for row in retained_classes:
    major_code = int(row['major_class'])
    major_label = row['major_label']

    df_class = df_corpus[df_corpus['major_class'] == major_code].copy().reset_index(drop=True)

    docs = df_class['text_clean'].tolist()
    if len(docs) < int(CONFIG['min_docs_total']):
        print(f"Skip major {major_code} (too few docs): n={len(docs)}")
        continue

    print(f"Fitting BERTopic: major {major_code} — {major_label} | n={len(docs)}")

    embeddings = embedder.encode(
        docs,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )

    umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine', random_state=SEED)
    hdbscan_model = hdbscan.HDBSCAN(
        min_cluster_size=int(CONFIG['bertopic_min_topic_size']),
        metric='euclidean',
        cluster_selection_method='eom',
        prediction_data=True,
    )

    vectorizer_model = CountVectorizer(
        stop_words='english',
        ngram_range=tuple(CONFIG['bertopic_n_gram_range']),
    )

    topic_model = BERTopic(
        umap_model=umap_model,
        hdbscan_model=hdbscan_model,
        vectorizer_model=vectorizer_model,
        top_n_words=int(CONFIG['bertopic_top_n_words']),
        calculate_probabilities=True,
        verbose=False,
    )

    topics, probs = topic_model.fit_transform(docs, embeddings)

    class_models[major_code] = topic_model
    class_docs[major_code] = df_class
    class_embeddings[major_code] = embeddings

    # Per-topic summary
    topic_info = topic_model.get_topic_info()
    for _, trow in topic_info.iterrows():
        topic_id = int(trow['Topic'])
        if topic_id == -1:
            top_terms = ''
        else:
            top_terms = _topic_top_terms(topic_model, topic_id, top_n=int(CONFIG['bertopic_top_n_words']))

        per_topic_rows.append({
            'major_class': major_code,
            'major_label': major_label,
            'topic_id': topic_id,
            'topic_size': int(trow.get('Count', 0)),
            'top_terms': top_terms,
        })

    # Per-doc assignments
    for i, doc_topic in enumerate(topics):
        probs_i = probs[i] if probs is not None and len(probs) > i else None
        per_doc_rows.append({
            'doc_id': df_class.loc[i, 'doc_id'],
            'source': df_class.loc[i, 'source'],
            'year': int(df_class.loc[i, 'year']) if not pd.isna(df_class.loc[i, 'year']) else None,
            'major_class': major_code,
            'major_label': major_label,
            'topic_id': int(doc_topic),
            'topic_prob': _safe_max_topic_prob(probs_i),
            'text_snippet': (df_class.loc[i, 'text_raw'][:280] if isinstance(df_class.loc[i, 'text_raw'], str) else ''),
        })

per_doc_topics = pd.DataFrame(per_doc_rows)
per_topic_summary = pd.DataFrame(per_topic_rows)

print('Models fit:', len(class_models))
display(per_topic_summary.head(10))
display(per_doc_topics.head(5))

## 7) Quantify “Organic Separation” (topic prevalence by source; JSD)

Within each major class, compute topic prevalence distributions for patents vs scopus and quantify divergence using Jensen–Shannon divergence:

$$JSD(p,q)=\tfrac12 KL(p\|m)+\tfrac12 KL(q\|m),\; m=\tfrac12(p+q).$$

Also identify topics with strong source skew.

In [ ]:
def _kl_div(p: np.ndarray, q: np.ndarray) -> float:
    # Assumes p,q are valid probability vectors; uses natural log.
    return float(np.sum(p * np.log(p / q)))


def jsd(p: np.ndarray, q: np.ndarray, *, eps: float) -> float:
    p = np.asarray(p, dtype=float)
    q = np.asarray(q, dtype=float)

    # Smooth + renormalize
    p = np.clip(p, eps, None)
    q = np.clip(q, eps, None)
    p = p / p.sum()
    q = q / q.sum()

    m = 0.5 * (p + q)
    return 0.5 * _kl_div(p, m) + 0.5 * _kl_div(q, m)


jsd_rows: list[dict] = []
skew_rows: list[dict] = []

if len(per_doc_topics) == 0:
    print('No per-doc topics available (did BERTopic run?).')
else:
    eps = float(CONFIG['jsd_epsilon'])

    for major_code in sorted(per_doc_topics['major_class'].unique()):
        df_m = per_doc_topics[per_doc_topics['major_class'] == major_code].copy()
        major_label = df_m['major_label'].iloc[0] if len(df_m) else ''

        # Topic prevalence per source
        tab = (
            df_m
            .groupby(['source', 'topic_id'])
            .size()
            .rename('count')
            .reset_index()
        )

        pivot = tab.pivot_table(index='topic_id', columns='source', values='count', fill_value=0)
        for col in ['patent', 'scopus']:
            if col not in pivot.columns:
                pivot[col] = 0

        p = (pivot['patent'].values + eps)
        q = (pivot['scopus'].values + eps)
        p = p / p.sum()
        q = q / q.sum()

        jsd_val = jsd(p, q, eps=eps)
        jsd_rows.append({'major_class': int(major_code), 'major_label': major_label, 'jsd': jsd_val})

        # Per-topic skew stats
        totals = pivot[['patent', 'scopus']].sum(axis=1).replace(0, np.nan)
        pivot['p_patent'] = pivot['patent'] / totals
        pivot['p_scopus'] = pivot['scopus'] / totals
        pivot = pivot.fillna(0)
        pivot['diff_patent_minus_scopus'] = pivot['p_patent'] - pivot['p_scopus']

        for topic_id, prow in pivot.reset_index().iterrows():
            tid = int(prow['topic_id'])
            skew_rows.append({
                'major_class': int(major_code),
                'major_label': major_label,
                'topic_id': tid,
                'patent_count': int(prow['patent']),
                'scopus_count': int(prow['scopus']),
                'p_patent_within_topic': float(prow['p_patent']),
                'diff_patent_minus_scopus': float(prow['diff_patent_minus_scopus']),
            })

jsd_df = pd.DataFrame(jsd_rows).sort_values('jsd', ascending=False)
skew_df = pd.DataFrame(skew_rows)

print('JSD per major class (higher = stronger source separation across topics):')
display(jsd_df)

# Join skew back into per_topic_summary
per_topic_summary = per_topic_summary.merge(
    skew_df,
    on=['major_class', 'major_label', 'topic_id'],
    how='left',
)

# Also join per-class JSD into per_topic_summary
per_topic_summary = per_topic_summary.merge(
    jsd_df,
    on=['major_class', 'major_label'],
    how='left',
)

# Show most skewed topics overall
if len(per_topic_summary):
    display(
        per_topic_summary
        .sort_values('diff_patent_minus_scopus', ascending=False)
        [['major_class', 'major_label', 'topic_id', 'topic_size', 'patent_count', 'scopus_count', 'diff_patent_minus_scopus', 'top_terms']]
        .head(20)
    )

## 8) Per-Class Topic Inspection (top terms, representative docs, outliers)

Generate per-class inspection tables:
- topic size + top terms
- representative doc snippets
- outlier topic counts (`topic_id = -1`)

Also sample docs per topic stratified by source.

In [ ]:
def inspect_class_topics(major_code: int, n_topics_show: int = 12, samples_per_source: int = 2) -> None:
    if major_code not in class_models:
        print(f"No model for major_class={major_code}")
        return

    model = class_models[major_code]

    df_docs = per_doc_topics[per_doc_topics['major_class'] == major_code].copy()
    if len(df_docs) == 0:
        print(f"No docs for major_class={major_code}")
        return

    major_label = df_docs['major_label'].iloc[0]
    print(f"=== Major {major_code} — {major_label} ===")

    info = model.get_topic_info().copy()
    display(info.head(n_topics_show))

    # Build a compact per-topic table with top terms
    topic_ids = info['Topic'].astype(int).tolist()[:n_topics_show]
    rows = []
    for tid in topic_ids:
        rows.append({
            'topic_id': tid,
            'size': int(info.loc[info['Topic'] == tid, 'Count'].iloc[0]) if (info['Topic'] == tid).any() else None,
            'top_terms': _topic_top_terms(model, tid, top_n=int(CONFIG['bertopic_top_n_words'])) if tid != -1 else '',
        })

    display(pd.DataFrame(rows))

    # Outlier count
    outliers = int((df_docs['topic_id'] == -1).sum())
    print('Outliers (-1 topic):', outliers)

    # Sample docs per topic stratified by source
    for tid in topic_ids:
        if tid == -1:
            continue
        print(f"\nTopic {tid}: {_topic_top_terms(model, tid, top_n=10)}")

        df_t = df_docs[df_docs['topic_id'] == tid]
        if len(df_t) == 0:
            continue

        for src in ['patent', 'scopus']:
            df_s = df_t[df_t['source'] == src]
            if len(df_s) == 0:
                continue
            sample = df_s.sample(n=min(samples_per_source, len(df_s)), random_state=SEED)
            print(f"  {src} examples:")
            for _, r in sample.iterrows():
                snip = (r.get('text_snippet') or '').replace('\n', ' ')
                print('   -', snip[:220])


# Inspect the top-JSD class (if any)
if 'jsd_df' in globals() and len(jsd_df):
    top_major = int(jsd_df.iloc[0]['major_class'])
    inspect_class_topics(top_major)
else:
    print('Run Section 6/7 first to inspect topics.')

## 9) Cross-Source Comparison Inside Class (patent-dominant vs scopus-dominant topics)

Label topics as patent-dominant / scopus-dominant / mixed based on prevalence thresholds, and produce side-by-side summaries with example documents. Also compute per-topic average year by source (optional temporal signal).

In [ ]:
def dominance_label(p_patent: float, *, thr: float) -> str:
    if p_patent >= thr:
        return 'patent-dominant'
    if (1.0 - p_patent) >= thr:
        return 'scopus-dominant'
    return 'mixed'


thr = float(CONFIG['dominance_threshold'])

if len(per_topic_summary) == 0:
    print('No per-topic summary available.')
else:
    per_topic_summary['dominance'] = per_topic_summary['p_patent_within_topic'].fillna(0.5).map(
        lambda x: dominance_label(float(x), thr=thr)
    )

    # Average year by topic+source (pivoted to keep one row per topic)
    avg_year = (
        per_doc_topics
        .dropna(subset=['year'])
        .groupby(['major_class', 'topic_id', 'source'])
        ['year']
        .mean()
        .rename('avg_year')
        .reset_index()
        .pivot_table(index=['major_class', 'topic_id'], columns='source', values='avg_year')
        .reset_index()
        .rename(columns={'patent': 'avg_year_patent', 'scopus': 'avg_year_scopus'})
    )

    per_topic_summary = per_topic_summary.merge(avg_year, on=['major_class', 'topic_id'], how='left')

    # Show a compact table for the most separated class (top JSD)
    if 'jsd_df' in globals() and len(jsd_df):
        major_code = int(jsd_df.iloc[0]['major_class'])
        df_show = per_topic_summary[per_topic_summary['major_class'] == major_code].copy()
        df_show = df_show.sort_values(['dominance', 'topic_size'], ascending=[True, False])

        display(
            df_show[[
                'major_class', 'major_label', 'topic_id', 'topic_size',
                'patent_count', 'scopus_count', 'p_patent_within_topic',
                'dominance', 'avg_year_patent', 'avg_year_scopus', 'top_terms'
            ]].head(30)
        )

        # Side-by-side examples for dominant topics
        dom_topics = (
            df_show[df_show['dominance'].isin(['patent-dominant', 'scopus-dominant'])]
            ['topic_id']
            .dropna()
            .astype(int)
            .tolist()[:6]
        )
        for tid in dom_topics:
            label = df_show[df_show['topic_id'] == tid]['dominance'].iloc[0]
            print(
                f"\nMajor {major_code} Topic {tid} ({label}): "
                f"{df_show[df_show['topic_id'] == tid]['top_terms'].iloc[0]}"
            )
            df_t = per_doc_topics[(per_doc_topics['major_class'] == major_code) & (per_doc_topics['topic_id'] == tid)]
            for src in ['patent', 'scopus']:
                df_s = df_t[df_t['source'] == src]
                if len(df_s) == 0:
                    continue
                sample = df_s.sample(n=min(2, len(df_s)), random_state=SEED)
                print(f"  {src} examples:")
                for _, r in sample.iterrows():
                    print('   -', (r.get('text_snippet') or '')[:220].replace('\n', ' '))
    else:
        print('Run Sections 6/7 to populate JSD + dominance labels.')

## 10) Visualizations (UMAP scatter, topic bars, heatmaps)

Create per-class visual outputs:
1) UMAP scatter of document embeddings colored by source and shaped by topic
2) Topic prevalence bar charts split by source
3) Heatmap of topics (rows) vs sources (cols) showing normalized prevalence
4) Optional BERTopic interactive visualizations saved to HTML

In [ ]:
from umap import UMAP


def visualize_major_class(major_code: int, *, max_points: int = 3000) -> dict[str, Path]:
    if major_code not in class_models:
        print(f"No BERTopic model for major_class={major_code}")
        return {}

    model = class_models[major_code]
    df_docs = per_doc_topics[per_doc_topics['major_class'] == major_code].copy().reset_index(drop=True)
    df_src = class_docs.get(major_code)
    emb = class_embeddings.get(major_code)

    if df_src is None or emb is None:
        print(f"Missing cached docs/embeddings for major_class={major_code}")
        return {}

    major_label = df_docs['major_label'].iloc[0] if len(df_docs) else ''

    out_dir = OUT_DIR / f"major_{major_code}"
    out_dir.mkdir(parents=True, exist_ok=True)

    # Subsample for plotting
    if len(df_docs) > max_points:
        idx = np.random.RandomState(SEED).choice(len(df_docs), size=max_points, replace=False)
        dfp = df_docs.iloc[idx].copy().reset_index(drop=True)
        embp = emb[idx]
    else:
        dfp = df_docs
        embp = emb

    # UMAP 2D on embeddings
    reducer = UMAP(n_neighbors=15, n_components=2, min_dist=0.05, metric='cosine', random_state=SEED)
    xy = reducer.fit_transform(embp)
    plot_df = dfp.copy()
    plot_df['x'] = xy[:, 0]
    plot_df['y'] = xy[:, 1]

    # 1) UMAP scatter (color=source, marker=topic)
    markers = ['o', 's', '^', 'v', 'D', 'P', 'X', '*']
    plot_df['topic_marker'] = plot_df['topic_id'].map(lambda t: markers[int(t) % len(markers)] if int(t) >= 0 else 'x')

    plt.figure(figsize=(7, 6))
    # plot per marker so matplotlib can render shapes
    for m in sorted(plot_df['topic_marker'].unique()):
        dsm = plot_df[plot_df['topic_marker'] == m]
        sns.scatterplot(data=dsm, x='x', y='y', hue='source', marker=m, s=25, alpha=0.6, legend=False)
    plt.title(f"UMAP embeddings — Major {major_code} ({major_label})\ncolor=source, marker=topic")
    plt.tight_layout()
    scatter_path = out_dir / f"goal7_umap_major_{major_code}.png"
    plt.savefig(scatter_path, dpi=160)
    plt.show()

    # 2) Topic prevalence bars split by source
    prev = (
        df_docs
        .groupby(['topic_id', 'source'])
        .size()
        .rename('count')
        .reset_index()
    )

    plt.figure(figsize=(10, 4))
    sns.barplot(data=prev[prev['topic_id'] != -1], x='topic_id', y='count', hue='source')
    plt.title(f"Topic prevalence by source — Major {major_code} ({major_label})")
    plt.tight_layout()
    bars_path = out_dir / f"goal7_topic_prevalence_major_{major_code}.png"
    plt.savefig(bars_path, dpi=160)
    plt.show()

    # 3) Heatmap of normalized prevalence
    heat = prev.pivot_table(index='topic_id', columns='source', values='count', fill_value=0)
    heat = heat.sort_index()
    heat_norm = heat.div(heat.sum(axis=1).replace(0, np.nan), axis=0).fillna(0)

    plt.figure(figsize=(4, max(3, 0.25 * len(heat_norm))))
    sns.heatmap(heat_norm, annot=False, cmap='viridis')
    plt.title(f"Normalized topic prevalence (rows sum to 1)\nMajor {major_code}")
    plt.tight_layout()
    heat_path = out_dir / f"goal7_topic_heatmap_major_{major_code}.png"
    plt.savefig(heat_path, dpi=160)
    plt.show()

    # 4) Topics over time (PNG)
    over_time_path: Path | None = None
    try:
        df_year = df_docs.dropna(subset=['year']).copy()
        df_year['year'] = pd.to_numeric(df_year['year'], errors='coerce').astype('Int64')
        df_year = df_year.dropna(subset=['year']).copy()
        df_year = df_year[df_year['topic_id'] != -1].copy()

        if df_year['year'].nunique() >= 2 and len(df_year):
            top_topics = (
                df_year.groupby('topic_id').size().sort_values(ascending=False).head(8).index.tolist()
            )
            df_plot = df_year[df_year['topic_id'].isin(top_topics)].copy()
            yearly = (
                df_plot.groupby(['year', 'topic_id']).size().rename('count').reset_index()
            )

            plt.figure(figsize=(10, 4))
            sns.lineplot(data=yearly, x='year', y='count', hue='topic_id', marker='o')
            plt.title(f"Topics over time (top {len(top_topics)}) — Major {major_code} ({major_label})")
            plt.tight_layout()
            over_time_path = out_dir / f"goal7_topics_over_time_major_{major_code}.png"
            plt.savefig(over_time_path, dpi=160)
            plt.show()
        else:
            print(f'Skipping topics-over-time: insufficient year coverage for major_class={major_code}.')
    except Exception as e:
        print('topics-over-time plot failed (skipping):', e)
        over_time_path = None

    # 5) Optional BERTopic interactive artifacts (HTML + PNG)
    # These can fail for small/noisy classes where HDBSCAN assigns everything to outlier (-1)
    # or when there are too few non-outlier topics to build a global topic map.
    extra_paths: dict[str, Path] = {}
    non_outlier_topics = 0
    try:
        info = model.get_topic_info()
        if 'Topic' in info.columns:
            non_outlier_topics = int((info['Topic'] != -1).sum())
    except Exception as e:
        print('get_topic_info failed (skipping interactive HTML):', e)
        non_outlier_topics = 0

    if non_outlier_topics >= 2:
        try:
            fig = model.visualize_topics()
            p = out_dir / f"goal7_bertopic_topics_major_{major_code}.html"
            fig.write_html(str(p))
            extra_paths['bertopic_topics_html'] = p
        except Exception as e:
            print('visualize_topics failed (skipping):', e)
    else:
        print(f'Skipping visualize_topics: only {non_outlier_topics} non-outlier topic(s) for major_class={major_code}.')

    if non_outlier_topics >= 1:
        try:
            fig = model.visualize_barchart(top_n_topics=12)
            p = out_dir / f"goal7_bertopic_barchart_major_{major_code}.png"
            # Requires kaleido (installed in the Colab setup cell)
            fig.write_image(str(p), scale=2)
            extra_paths['bertopic_barchart_png'] = p
        except Exception as e:
            print('visualize_barchart export failed (skipping):', e)
    else:
        print(f'Skipping visualize_barchart: no non-outlier topics for major_class={major_code}.')

    out = {
        'scatter_png': scatter_path,
        'bars_png': bars_path,
        'heatmap_png': heat_path,
        **({'topics_over_time_png': over_time_path} if over_time_path is not None else {}),
        **extra_paths,
    }

    print('Saved artifacts:', {k: str(v) for k, v in out.items()})
    return out


print('Visualization helper loaded. Use the next cell to opt-in and generate PNG/HTML artifacts, or call visualize_major_class(<major_class_code>) directly.')


In [ ]:
# --- Optional (opt-in) visualization run ---
# This cell is intentionally disabled by default.
RUN_VISUALIZATIONS = False  # set True to generate PNG/HTML artifacts under outputs/goal7/

MAJOR_CLASSES_TO_VISUALIZE = [1, 2, 3, 4, 5]  # requested loop set

if RUN_VISUALIZATIONS:
    if not class_models:
        print('No class models available (did topic modeling run?)')
    else:
        for major_code in MAJOR_CLASSES_TO_VISUALIZE:
            print('\n' + '=' * 80)
            print('Visualizing major class:', major_code)
            artifacts = visualize_major_class(int(major_code))
            # show returned paths in the cell output
            artifacts

## 11) Export Artifacts (Scopus + Patents CSVs)

Write exactly two **CSV outputs** to the Goal #7 output folder:
- `goal7_scopus_topic_subclassified.csv`
- `goal7_patents_topic_subclassified.csv`

Each file includes a new **Primary Major Subclass** column derived from the topic model (within the Primary Major Class).

In [ ]:
out_dir = Path(CONFIG['out_dir'])
out_dir.mkdir(parents=True, exist_ok=True)

scopus_out = out_dir / 'goal7_scopus_topic_subclassified.csv'
patents_out = out_dir / 'goal7_patents_topic_subclassified.csv'

SUBCLASS_COL = 'Primary Major Subclass'

if len(per_doc_topics) == 0:
    raise RuntimeError('No per-doc topic assignments found. Run Section 6 (topic modeling) first.')

# Build a stable per-(class,topic) label from top terms
if len(per_topic_summary) == 0 or 'top_terms' not in per_topic_summary.columns:
    topic_labels = (
        per_doc_topics[['major_class', 'major_label', 'topic_id']]
        .drop_duplicates()
        .assign(top_terms='')
    )
else:
    topic_labels = (
        per_topic_summary[['major_class', 'major_label', 'topic_id', 'top_terms']]
        .drop_duplicates(subset=['major_class', 'topic_id'])
        .copy()
    )


def _subclass_label(topic_id: int, top_terms: str) -> str:
    if int(topic_id) == -1:
        return 'Outlier/Other'
    terms = (top_terms or '').strip()
    if terms:
        return f"Topic {int(topic_id)}: {terms}"
    return f"Topic {int(topic_id)}"


topic_labels['topic_subclass'] = [
    _subclass_label(int(tid), tt)
    for tid, tt in zip(topic_labels['topic_id'].tolist(), topic_labels['top_terms'].tolist())
]

per_doc_enriched = per_doc_topics.merge(
    topic_labels[['major_class', 'topic_id', 'topic_subclass']],
    on=['major_class', 'topic_id'],
    how='left',
)


def export_source(raw_df: pd.DataFrame, *, source: str, out_path: Path) -> pd.DataFrame:
    if source not in {'scopus', 'patent'}:
        raise ValueError(f"Unexpected source: {source}")

    df = raw_df.copy()
    df['doc_id'] = source + ':' + df['Serial Number'].fillna('').astype(str)

    df = df.merge(
        per_doc_enriched[['doc_id', 'topic_subclass']],
        on='doc_id',
        how='left',
    )

    df = df.rename(columns={'topic_subclass': SUBCLASS_COL})
    df[SUBCLASS_COL] = df[SUBCLASS_COL].fillna('Not modeled')

    df.to_csv(out_path, index=False)
    print(f"Wrote: {out_path} | rows={len(df)}")
    return df


scopus_with_subclass = export_source(scopus_raw, source='scopus', out_path=scopus_out)
patents_with_subclass = export_source(patents_raw, source='patent', out_path=patents_out)

# Quick sanity (Colab)
print('Subclass value counts (top 5):')
print('  Scopus:')
print(scopus_with_subclass[SUBCLASS_COL].value_counts().head(5))
print('  Patents:')
print(patents_with_subclass[SUBCLASS_COL].value_counts().head(5))

## 12) Optional: LDA Baseline per Class (sanity check) + Coherence

Optionally fit an LDA model per class as a simpler baseline, compute a coherence proxy (UMass), and see whether the patent-vs-scopus separation signals persist under this simpler topic model.

In [ ]:
from scipy import sparse


def top_words_from_components(components: np.ndarray, feature_names: list[str], top_n: int) -> list[list[str]]:
    topics = []
    for k in range(components.shape[0]):
        inds = np.argsort(components[k])[::-1][:top_n]
        topics.append([feature_names[i] for i in inds])
    return topics


def umass_coherence(topic_words: list[str], dtm_bin: sparse.spmatrix, vocab_index: dict[str, int]) -> float:
    # UMass coherence proxy using document co-occurrence counts.
    # dtm_bin is binary document-term matrix (docs x terms).
    eps = 1.0

    # Precompute doc frequencies for each term
    term_ids = [vocab_index[w] for w in topic_words if w in vocab_index]
    if len(term_ids) < 2:
        return float('nan')

    df = np.asarray(dtm_bin[:, term_ids].sum(axis=0)).ravel()

    score = 0.0
    pairs = 0
    for i in range(1, len(term_ids)):
        for j in range(0, i):
            wi = term_ids[i]
            wj = term_ids[j]
            # D(wi, wj)
            co = dtm_bin[:, [wi, wj]].min(axis=1).sum()
            co = float(co)
            dj = float(dtm_bin[:, wj].sum())
            score += math.log((co + eps) / (dj + eps))
            pairs += 1

    return float(score / max(pairs, 1))


RUN_LDA = bool(CONFIG.get('run_lda_baseline', False))
if not RUN_LDA:
    print('LDA baseline disabled. Set CONFIG[run_lda_baseline]=True to run.')
else:
    lda_rows: list[dict] = []

    for row in retained_classes:
        major_code = int(row['major_class'])
        major_label = row['major_label']

        df_class = df_corpus[df_corpus['major_class'] == major_code].copy().reset_index(drop=True)
        docs = df_class['text_clean'].tolist()
        if len(docs) < int(CONFIG['min_docs_total']):
            continue

        print(f"\nRunning LDA: major {major_code} — {major_label} | n={len(docs)}")

        vect = CountVectorizer(
            stop_words='english',
            ngram_range=(1, 2),
            min_df=2,
            max_df=0.95,
        )
        X = vect.fit_transform(docs)

        lda = LatentDirichletAllocation(
            n_components=int(CONFIG['lda_n_topics']),
            random_state=SEED,
            learning_method='batch',
        )
        doc_topics = lda.fit_transform(X)

        feature_names = vect.get_feature_names_out().tolist()
        topics = top_words_from_components(lda.components_, feature_names, top_n=12)

        # Coherence
        X_bin = (X > 0).astype(int)
        vocab_index = {w: i for i, w in enumerate(feature_names)}
        coherences = [umass_coherence(tw, X_bin, vocab_index) for tw in topics]

        for k, (words, coh) in enumerate(zip(topics, coherences)):
            lda_rows.append({
                'major_class': major_code,
                'major_label': major_label,
                'lda_topic_id': k,
                'top_terms': ', '.join(words),
                'umass_coherence': float(coh) if not (coh is None or np.isnan(coh)) else None,
            })

        display(pd.DataFrame(lda_rows).query('major_class == @major_code').head(10))

    lda_summary = pd.DataFrame(lda_rows)
    lda_out = OUT_DIR / 'goal7_lda_topic_summary.csv'
    lda_summary.to_csv(lda_out, index=False)
    print('Wrote:', lda_out)